# 📈 Stock Signal Dashboard — Data Collection

## Objective
Download historical OHLCV (Open, High, Low, Close, Volume) stock price data
for 20 companies across 5 sectors using the yfinance API.
Store data in two structures — dictionary for EDA, flat dataframe for ML training.

---

## Dataset
- **Source:** Yahoo Finance (via yfinance library)
- **Tickers:** 20 stocks across 5 sectors
- **Frequency:** Daily
- **Period:** January 2019 — December 2024
- **Total rows:** ~30,000 (1,509 rows × 20 stocks)

---

## Stocks Covered

| Sector | Tickers |
|---|---|
| Technology | AAPL, MSFT, GOOGL, META, NVDA |
| E-commerce & Cloud | AMZN, NFLX, CRM, ADBE, ORCL |
| Finance | JPM, BAC, GS, V, MA |
| Healthcare | JNJ, PFE, UNH |
| Other | TSLA, WMT |

---

## Two Data Structures
- **`all_data`** → dictionary, one df per stock → used for EDA
- **`df_all`** → all 20 stocks combined → used for ML training

---

## Notebook Structure
1. Import libraries
2. Download data for all 20 stocks
3. Validate and inspect combined data
4. Save raw data to disk

> Raw data is never modified directly.
> All cleaning happens in the next notebook.

In [1]:
# ============================================================
# CELL 1 - Import Libraries
# ============================================================
# yfinance  : pulls live stock data from Yahoo Finance API
# pandas    : handles tabular data (like Excel but in Python)
# numpy     : fast numerical computations
# matplotlib: base plotting library for charts
# os        : handles file paths and directory creation
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os


# Show full output without truncation
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.6f}'.format)

print("All imports successful!")

All imports successful!


--------

## Step 2 - Download Data for 20 Stocks

We download 20 stocks across 5 sectors.
Each stock gets a Ticker column for identification
when all stocks are combined into one flat dataframe.

---

### Why 5 years?
- Too little data (1 year) - model does not learn enough patterns
- Too much data (20 years) - older market behavior may not reflect today
- 5 years = sweet spot, enough data and stays relevant

---

### Why stop at 2024-12-31?
- 2019 to 2024 - Training data, model learns from this
- 2025          - Testing data, model has never seen this
- 2026 onwards  - Live predictions via Streamlit app
- Keeping these separate avoids data leakage

---

### Why auto_adjust=True?
On August 31 2020, Apple did a 4-for-1 stock split.
Without adjustment the price appears to drop 74% overnight.
The model would learn a completely wrong pattern.
auto_adjust=True corrects all historical prices so splits
and dividends do not create fake crashes in our data.

---

### Why Ticker column?
Without it the model cannot tell Apple from JPMorgan.
A price of $150 means nothing without knowing which company it is.

---

### Two structures created:
- all_data - dictionary, one df per stock, used for EDA
- df_all   - flat combined dataframe, used for ML training

In [2]:
# ============================================================
# CELL 2 - Download 20 Stocks from Yahoo Finance
# ============================================================
# Loop through all tickers and download each one
# Add Ticker column to identify each stock in combined data
# Store in dictionary for EDA and flat dataframe for ML
# try/except ensures one failed ticker does not crash pipeline
# ============================================================

TICKERS = [
    # Technology
    'AAPL', 'MSFT', 'GOOGL', 'META', 'NVDA',
    # E-commerce and Cloud
    'AMZN', 'NFLX', 'CRM', 'ADBE', 'ORCL',
    # Finance
    'JPM', 'BAC', 'GS', 'V', 'MA',
    # Healthcare
    'JNJ', 'PFE', 'UNH',
    # Other
    'TSLA', 'WMT'
]

START_DATE = "2019-01-01"
END_DATE   = "2024-12-31"

# Structure 1 - dictionary, one dataframe per stock (for EDA)
all_data = {}

# Structure 2 - list to combine into flat dataframe (for ML)
df_list = []

print(f"Downloading {len(TICKERS)} stocks from Yahoo Finance...")
print(f"Period : {START_DATE} to {END_DATE}")
print(f"Stocks : {len(TICKERS)}\n")

for ticker in TICKERS:
    try:
        # Download OHLCV data for this ticker
        df = yf.download(
            ticker,
            start=START_DATE,
            end=END_DATE,
            auto_adjust=True,
            progress=False
        )

        # Flatten multi-level columns
        # yfinance returns ('Close','AAPL') - we want just 'Close'
        df.columns = [col[0] for col in df.columns]

        # Add Ticker column - critical for identifying rows
        # when all stocks are combined into one dataframe
        df['Ticker'] = ticker

        # Store in dictionary for EDA use
        all_data[ticker] = df

        # Add to list for combining into flat dataframe
        df_list.append(df)

        print(f"  {ticker:6} - {df.shape[0]} rows downloaded")

    except Exception as e:
        # If one ticker fails, skip it and continue
        # Do not let one failure crash the entire pipeline
        print(f"  {ticker:6} - Failed: {e}")

# Combine all stocks into one flat dataframe for ML training
df_all = pd.concat(df_list, axis=0)

# Sort by Ticker then Date for clean ordering
df_all = df_all.sort_values(['Ticker', 'Date']).reset_index()

print(f"\n{'='*45}")
print(f"All downloads complete!")
print(f"{'='*45}")
print(f"Total rows    : {df_all.shape[0]}")
print(f"Total columns : {df_all.shape[1]}")
print(f"Total stocks  : {df_all['Ticker'].nunique()}")
print(f"Date range    : {df_all['Date'].min().date()} to {df_all['Date'].max().date()}")
print(f"{'='*45}")
print(f"\nColumns in dataset : {list(df_all.columns)}")
print(f"\nFirst 5 rows of combined data:")
df_all.head()

Period : 2019-01-01 to 2024-12-31
Stocks : 20

  AAPL   - 1509 rows downloaded
  MSFT   - 1509 rows downloaded
  GOOGL  - 1509 rows downloaded
  META   - 1509 rows downloaded
  NVDA   - 1509 rows downloaded
  AMZN   - 1509 rows downloaded
  NFLX   - 1509 rows downloaded
  CRM    - 1509 rows downloaded
  ADBE   - 1509 rows downloaded
  ORCL   - 1509 rows downloaded
  JPM    - 1509 rows downloaded
  BAC    - 1509 rows downloaded
  GS     - 1509 rows downloaded
  V      - 1509 rows downloaded
  MA     - 1509 rows downloaded
  JNJ    - 1509 rows downloaded
  PFE    - 1509 rows downloaded
  UNH    - 1509 rows downloaded
  TSLA   - 1509 rows downloaded
  WMT    - 1509 rows downloaded

All downloads complete!
Total rows    : 30180
Total columns : 7
Total stocks  : 20
Date range    : 2019-01-02 to 2024-12-30

Columns in dataset : ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker']

First 5 rows of combined data:


,Date,Close,High,Low,Open,Volume,Ticker
0,2019-01-02,37.469204,37.689864,36.593688,36.750285,148158800,AAPL
1,2019-01-03,33.736988,34.574540,33.691907,34.161694,365248800,AAPL
2,2019-01-04,35.177197,35.246006,34.118988,34.292192,234428400,AAPL
3,2019-01-07,35.098900,35.312442,34.617248,35.281596,219111200,AAPL
4,2019-01-08,35.768009,36.021887,35.238905,35.485661,164101200,AAPL


## Step 3 - Validate and Inspect Combined Data

Before saving, we validate the combined dataset.
.

### What we check:
- Total rows and columns
- All 20 stocks are present
- No stock has significantly fewer rows than others
- Data types are correct
- No completely empty columns

In [3]:
# ============================================================
# CELL 3 - Validate Combined Dataset
# ============================================================
# Before saving we verify the data is complete and correct
# This catches problems early before they cause issues later
# ============================================================

print("DATASET VALIDATION REPORT")
print("="*45)

# Basic shape
print(f"\n1. Basic Info")
print(f"   Total rows    : {df_all.shape[0]}")
print(f"   Total columns : {df_all.shape[1]}")
print(f"   Column names  : {list(df_all.columns)}")

# Check all 20 stocks present
print(f"\n2. Stocks Present ({df_all['Ticker'].nunique()} total)")
print(f"   {sorted(df_all['Ticker'].unique())}")

# Check row count per stock
print(f"\n3. Rows Per Stock")
rows_per_stock = df_all.groupby('Ticker').size().sort_values()
print(rows_per_stock.to_string())

# Check data types
print(f"\n4. Data Types")
print(df_all.dtypes.to_string())

# Check for missing values
print(f"\n5. Missing Values Per Column")
missing = df_all.isnull().sum()
print(missing.to_string())

# Date range
print(f"\n6. Date Range")
print(f"   Start : {df_all['Date'].min().date()}")
print(f"   End   : {df_all['Date'].max().date()}")

print(f"\n{'='*45}")
print(f"Validation complete!")
print(f"{'='*45}")

DATASET VALIDATION REPORT

1. Basic Info
   Total rows    : 30180
   Total columns : 7
   Column names  : ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker']

2. Stocks Present (20 total)
   ['AAPL', 'ADBE', 'AMZN', 'BAC', 'CRM', 'GOOGL', 'GS', 'JNJ', 'JPM', 'MA', 'META', 'MSFT', 'NFLX', 'NVDA', 'ORCL', 'PFE', 'TSLA', 'UNH', 'V', 'WMT']

3. Rows Per Stock
Ticker
AAPL     1509
ADBE     1509
AMZN     1509
BAC      1509
CRM      1509
GOOGL    1509
GS       1509
JNJ      1509
JPM      1509
MA       1509
META     1509
MSFT     1509
NFLX     1509
NVDA     1509
ORCL     1509
PFE      1509
TSLA     1509
UNH      1509
V        1509
WMT      1509

4. Data Types
Date      datetime64[s]
Close           float64
High            float64
Low             float64
Open            float64
Volume            int64
Ticker              str

5. Missing Values Per Column
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
Ticker    0

6. Date Range
   Start : 2019-01-02
   End   : 2

## Step 4 - Save Raw Data to Disk

We save the combined dataset immediately after validation.

### Why save it?
- Avoid hitting the API every time we run the notebook
- Raw data is our single source of truth
- If something goes wrong in cleaning, we always have the original

### Why index=False here?
- Unlike single stock data, df_all already has Date as a regular column
- Not as an index - so we use index=False to avoid saving
  a meaningless 0,1,2 numeric index into the CSV

In [4]:
# ============================================================
# CELL 4 - Save Raw Data to Disk
# ============================================================
# Save df_all - all 20 stocks combined - to one CSV file
# index=False because Date is already a regular column
# This file is the foundation of the entire project
# Never modify this file directly
# ============================================================

# Save path
raw_data_path = "../data/raw/all_stocks_raw.csv"

# Save to CSV
df_all.to_csv(raw_data_path, index=False)

# Verify file saved correctly
file_size = os.path.getsize(raw_data_path)

print("Raw data saved successfully!")
print("="*45)
print(f"Location  : {raw_data_path}")
print(f"Rows      : {df_all.shape[0]}")
print(f"Columns   : {df_all.shape[1]}")
print(f"Stocks    : {df_all['Ticker'].nunique()}")
print(f"File size : {file_size / 1024:.1f} KB")
print("="*45)
print("\nThis file is RAW data - never modify it directly!")

Raw data saved successfully!
Location  : ../data/raw/all_stocks_raw.csv
Rows      : 30180
Columns   : 7
Stocks    : 20
File size : 2890.6 KB

This file is RAW data - never modify it directly!
